# LLM Inference

## Project setup

In [1]:
from pathlib import Path
import sys
import os

# Find project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print("Project root:")
print(PROJECT_ROOT)

Project root:
/Users/roshanjaiswal/Downloads/project/finresearch-qwen


## Configuration

In [ ]:
import os

os.environ["MODEL_NAME"] = "Qwen/Qwen3-0.6B"
os.environ["ADAPTER_DIR"] = str(PROJECT_ROOT/ "artifacts"/"finresearch-qwen")
os.environ["MAX_NEW_TOKENS"] = "512"

print("MODEL_NAME:", os.environ["MODEL_NAME"])
print("ADAPTER_DIR:",os.environ["ADAPTER_DIR"])

MODEL_NAME: Qwen/Qwen3-4B
ADAPTER_DIR: /Users/roshanjaiswal/Downloads/project/finresearch-qwen/artifacts/finresearch-qwen


## Verify GPU

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("BF16 supported:", torch.cuda.is_bf16_supported())

else:
    print("No CUDA GPU detected.")

CUDA available: False
No CUDA GPU detected.


## Verify files

In [5]:
from pathlib import Path

adapter_dir = (PROJECT_ROOT/ "artifacts"/"finresearch-qwen")
print("Adapter directory:")
print(adapter_dir)
print("\nExists:", adapter_dir.exists())

if adapter_dir.exists():
    print("\nFiles:")
    for file in sorted(adapter_dir.iterdir()):
        print(" -",file.name)
        

# adapter_config.json
# adapter_model.safetensors

Adapter directory:
/Users/roshanjaiswal/Downloads/project/finresearch-qwen/artifacts/finresearch-qwen

Exists: False


## Import Inference

In [6]:
from src.inference.generate import (
    generate,
    load_model,
)

print("Inference module loaded successfully.")

/Users/roshanjaiswal/Downloads/project/finresearch-qwen/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 14:35:20.362000 40907 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Inference module loaded successfully.


## Load model

In [ ]:
tokenizer, model = load_model()

print()
print("Model loaded successfully.")
print("Model device:", model.device)

## Basic inference

In [ ]:
question = """
What is the difference between revenue growth
and profit growth for a company?
"""

answer = generate(
    question
)

print(answer)

#This will test fine- tuned model without RAG

## Financial calculation test

In [ ]:
question = """
A company reported revenue of ₹10,000 crore in FY2024
and ₹12,500 crore in FY2025.

Calculate the year-over-year revenue growth.
Show the formula and calculation.
"""

answer = generate(
    question
)

print(answer)

## RAG-style inference

In [ ]:
context = """
Reliance Industries reported revenue of ₹9,74,864 crore
for FY2024 and ₹10,71,174 crore for FY2025.
"""

question = """
Calculate the revenue growth from FY2024 to FY2025.
Show the calculation and explain the result.
"""

answer = generate(
    question,
    context=context
)

print(answer)

## Compare without and with context

In [ ]:
question = """
What was the company's revenue growth between FY2024
and FY2025? Show the calculation.
"""

context = """
Revenue FY2024: ₹9,74,864 crore
Revenue FY2025: ₹10,71,174 crore
"""

print("=" * 70)
print("WITHOUT RETRIEVED CONTEXT")
print("=" * 70)

answer_without_rag = generate(
    question
)

print(answer_without_rag)


print()
print("=" * 70)
print("WITH RETRIEVED CONTEXT")
print("=" * 70)

answer_with_rag = generate(
    question,
    context
)

print(answer_with_rag)

## Create a reusable helper

In [8]:
def ask_finresearch(
    question,
    context=""
):

    print("=" * 80)
    print("QUESTION")
    print("=" * 80)

    print(question)

    if context:

        print()
        print("=" * 80)
        print("RETRIEVED EVIDENCE")
        print("=" * 80)

        print(context)

    print()
    print("=" * 80)
    print("FINRESEARCH-QWEN")
    print("=" * 80)

    answer = generate(
        question,
        context
    )

    print(answer)

    return answer

In [ ]:
ask_finresearch(
    """
    Calculate the revenue growth between FY2024 and FY2025.
    Show the formula.
    """,
    """
    FY2024 revenue: ₹9,74,864 crore
    FY2025 revenue: ₹10,71,174 crore
    """
)

## Test multiple financial questions

In [ ]:
questions = [

    """
    What is revenue growth?
    Explain the calculation with an example.
    """,

    """
    A company has revenue of ₹20,000 crore
    and profit of ₹2,000 crore.
    Calculate the profit margin.
    """,

    """
    Revenue increased from ₹50,000 crore to ₹60,000 crore.
    Calculate the percentage increase.
    """,

    """
    Explain the difference between EBITDA margin
    and net profit margin.
    """,

]

In [ ]:
for i, question in enumerate(
    questions,
    start=1
):

    print()
    print("#" * 80)
    print(f"QUESTION {i}")
    print("#" * 80)

    answer = generate(
        question
    )

    print(answer)